# Sprint 1 - Otras acciones de procesamiento de datos
Autor: Adrián Robles Arques



Cada evento Event contiene un atributo type_id que identifica el tipo de evento (En este ejercicio, nos centraremos en los pases, que tienen type_id="1"). El resto de atributos de los campos Event de interés se detallan a continuación: 

    period_id - Primera (1) o segunda (2) parte del partido
    min - Minuto del partido en el que se produjo el evento 
    sec - Segundo del partido en el que se produjo el evento
    outcome - En los pases, outcome="1" indica que el pase fue exitoso 
    x - Valor de la coordenada x en el campo donde se produjo el evento
    y - Valor de la coordenada y en el campo donde se produjo el evento 

Dentro de cada evento, existen una serie de cualificadores Q que proporcionan información detallada de cada uno de los eventos. Para este ejercicio, nos centraremos en el atributo qualifier_id, concretamente en dos valores: 

 

    qualifier_id = "140" – Coordenada x final del pase 
    qualifier_id = "141" – Coordenada y final del pase 


SE PIDE:

A partir del fichero OptaF24.xml almacenado en un servidor FTP, que contiene los eventos de un partido concreto, crear un dataframe que contenga todos los pases de dicho partido. Este dataframe deberá contener los siguientes campos (columnas):



Para ello: 

1. Descarga el archivo OptaF24.xml creando una conexión al servidor FTP 

 

    Servidor: f31-preview.runhosting.com 
    Usuario: 4009006_DATOS 
    Contraseña: Rafa9999 

 

2. Crea una lista inicialmente vacía para cada de las columnas del dataframe 

 

3. Itera cada evento y filtra aquellos que correspondan a pases (atributo type_id="1"), añadiendo a cada lista del paso anterior los atributos correspondientes de cada evento. Para obtener el valor de un atributo puedes utilizar la función attrib.get() 

In [2]:
import ftplib

In [ ]:
# Nos conectamos al servidor FTP
ftp = ftplib.FTP('f31-preview.runhosting.com', '4009006_DATOS', 'Rafa9999')

In [5]:
# Listamos los archivos en el directorio actual
archivos = ftp.nlst()

archivos

['.', '..', 'colors.json', 'OptaF24.xml', 'books.xml']

In [6]:
# Extraemos el archivo OptaF24

datos_futbol = 'OptaF24.xml'

ftp.retrbinary(f'RETR {datos_futbol}', open(datos_futbol, 'wb').write)

'226 Transfer complete'

In [7]:
# Cerramos la conexión al servidor FTP
ftp.quit()

'221 Goodbye.'

In [8]:
# Importamos la librería para manejar XML
import xml.etree.ElementTree as ET

In [14]:
# Visualizamos la estructura del XML
tree = ET.parse(datos_futbol)
root = tree.getroot()

# Visualizamos los niveles del XML
for child in root:
    print(child.tag, child.attrib)
    for subchild in child:
        print(subchild.tag, subchild.attrib)

Game {'id': '360481', 'away_team_id': '43', 'away_team_name': 'Manchester City', 'competition_id': '8', 'competition_name': 'English Barclays Premier League', 'game_date': '2011-08-21T16:00:00', 'home_team_id': '30', 'home_team_name': 'Bolton Wanderers', 'matchday': '2', 'period_1_start': '2011-08-21T16:00:38', 'period_2_start': '2011-08-21T17:03:47', 'season_id': '2011', 'season_name': 'Season 2011/2012'}
Event {'id': '301038339', 'event_id': '1', 'type_id': '34', 'period_id': '16', 'min': '0', 'sec': '0', 'team_id': '43', 'outcome': '1', 'x': '0.0', 'y': '0.0', 'timestamp': '2011-08-21T15:23:06.696', 'last_modified': '2011-08-21T15:54:56'}
Event {'id': '1475524684', 'event_id': '1', 'type_id': '34', 'period_id': '16', 'min': '0', 'sec': '0', 'team_id': '30', 'outcome': '1', 'x': '0.0', 'y': '0.0', 'timestamp': '2011-08-21T15:39:39.166', 'last_modified': '2011-08-21T16:06:40'}
Event {'id': '2036897618', 'event_id': '2', 'type_id': '32', 'period_id': '1', 'min': '0', 'sec': '0', 'team_

In [35]:
# Vamos a recorrer todos los elementos del XML
for element in root:
    print(element.tag, element.attrib)

Game {'id': '360481', 'away_team_id': '43', 'away_team_name': 'Manchester City', 'competition_id': '8', 'competition_name': 'English Barclays Premier League', 'game_date': '2011-08-21T16:00:00', 'home_team_id': '30', 'home_team_name': 'Bolton Wanderers', 'matchday': '2', 'period_1_start': '2011-08-21T16:00:38', 'period_2_start': '2011-08-21T17:03:47', 'season_id': '2011', 'season_name': 'Season 2011/2012'}


Tenemos que crear un Dataframe con los siguientes datos:

* Equipo - Equipo que realiza el pase
* Periodo - Primer o segundo tiempo
* Minuto - Momento cuando se realiza el pase
* Segundo - Segundo exacto del pase
* x_origen - Coordenada x donde se inicia el pase
* y_origen - Coordenada y donde se inicia el pase
* x_destino - Coordenada x donde se termina el pase
* y_destino - Coordenada y donde se termina el pase
* Resultado - Pase exitoso = 1, Pase fallido = 0

Todos los datos están en cada uno de los eventos (pase --> type_id = 1), salvo el nombre del equipo.
Para el nombre del equipo es necesario cotejar el team_id que aparece en el evento con los datos de away_team y home_team ID del partido, y extraer el nombre del equipo correspondiente.

In [40]:
# Extraemos los ids de todos los equipos y sus respectivos nombres
equipos = {}

for game in root:
    # Buscamos la id y el nombre del equipo visitante y el equipo local
    team_id = game.attrib.get('away_team_id')
    team_name = game.attrib.get('away_team_name')
    
    # Guardamos en el diccionario, si no existe
    equipos[team_id] = team_name
    
    #Buscamos la id y el nombre del equipo local
    team_id = game.attrib.get('home_team_id')
    team_name = game.attrib.get('home_team_name')
    
    # Guardamos en el diccionario, si no existe
    equipos[team_id] = team_name
    


In [44]:
for k, v in equipos.items():
    print(f'ID: {k}, Nombre: {v}')

ID: 43, Nombre: Manchester City
ID: 30, Nombre: Bolton Wanderers


Al parecer, en base a los resultados obtenidos, en el dataset solo hay datos de un único partido entre el Manchester City y el Bolton Wanderers, cosa que se puede corroborar con una inspección del archivo XML en cuestion. 

A continuación recorreremos todos los eventos extrayendo los datos de interés, teniendo en cuenta lo siguiente:
- Los pases tienen el valor "type_id=1"
- Los valores de coordenadas x e y finales están en los Q con "qualifier_id= 140" y "qualifier_id= 141" respectivamente.

In [57]:
# Creamos un diccionario para almacenar los eventos de interés
pases = {}

# Recorremos los eventos del XML para extraer los datos de interés
for games in root.iter():
    for event in games.iter():
        if event.attrib.get('type_id') == '1':  # Pases
            # Extraemos los datos de interés
            event_id = event.attrib.get('event_id')
            team_id = event.attrib.get('team_id')
            period = event.attrib.get('period_id')
            minute = event.attrib.get('min')
            second = event.attrib.get('sec')
            x_origin = event.attrib.get('x')
            y_origin = event.attrib.get('y')
            outcome = event.attrib.get('outcome')
            
            # Extraemos las coordenadas finales del pase
            for qualifier in event:
                if qualifier.attrib.get('qualifier_id') == '140':  # Coordenada X final
                    x_final = qualifier.attrib.get('value')
                elif qualifier.attrib.get('qualifier_id') == '141':  # Coordenada Y final
                    y_final = qualifier.attrib.get('value')
            
            # Almacenamos los datos en el diccionario
            pases[event_id] = {
                'team_id': team_id,
                'period': period,
                'minute': minute,
                'second': second,
                'x_origin': x_origin,
                'y_origin': y_origin,
                'x_final': x_final,
                'y_final': y_final,
                'outcome': outcome
            }

In [58]:
print(len(pases))  # Mostramos la cantidad de pases encontrados

704


In [59]:
# Importamos la librería para manejar DataFrames
import pandas as pd

In [63]:
df_pases = pd.DataFrame.from_dict(pases, orient='index')
df_pases.head()  # Mostramos las primeras filas del DataFrame

,team_id,period,minute,second,x_origin,y_origin,x_final,y_final,outcome
3,43,1,0,1,50.1,50.0,52.4,49.1,1
4,43,1,0,2,48.2,49.1,29.0,76.5,1
7,43,1,0,19,27.8,100.0,49.3,93.4,0
5,30,1,0,23,50.9,20.0,63.6,30.1,1
9,43,1,0,30,29.3,73.0,30.2,55.6,1


In [64]:
# Ajustamos los datos añadiendo los nombres de los equipos y renombrando columnas
df_pases['team_id'] = df_pases['team_id'].map(equipos)
df_pases.rename(columns={
    'x_origin': 'x_origen',
    'y_origin': 'y_origen',
    'x_final': 'x_final',
    'y_final': 'y_final',
    'team_id': 'equipo',
    'period': 'periodo',
    'minute': 'minuto',
    'second': 'segundo',
    'outcome': 'resultado'
}, inplace=True)
df_pases.reset_index(drop=True, inplace=True)

df_pases.head()  # Mostramos las primeras filas del DataFrame ajustado

,equipo,periodo,minuto,segundo,x_origen,y_origen,x_final,y_final,resultado
0,Manchester City,1,0,1,50.1,50.0,52.4,49.1,1
1,Manchester City,1,0,2,48.2,49.1,29.0,76.5,1
2,Manchester City,1,0,19,27.8,100.0,49.3,93.4,0
3,Bolton Wanderers,1,0,23,50.9,20.0,63.6,30.1,1
4,Manchester City,1,0,30,29.3,73.0,30.2,55.6,1


OPCIONAL: Obtén  los pases de más de 20 metros de avance en la coordenada X ¿Qué equipo ha efectuado más veces estos pases?

Según el sistema de coordenadas especificado en el ejercicio, la dirección de avance es la dirección positiva en el eje de las X para ambos equipos.
De modo que la longitud en el eje X del pase será $x_{origen} - x_{final}$.

In [66]:
# Creamos una nueva columna para calcular el avance en la coordenada X
df_pases['avance_x'] = df_pases['x_origen'].astype(float) - df_pases['x_final'].astype(float)

# Filtramos los pases de más de 20 metros de avance en la coordenada X
pases_largos = df_pases[df_pases['avance_x'] > 20]

# Hacemos un recuento por equipo de los pases largos
pases_largos_count = pases_largos['equipo'].value_counts()

# Mostramos los resultados
print(pases_largos_count)

equipo
Bolton Wanderers    6
Manchester City     5
Name: count, dtype: int64


Según los datos, el Bolton Wanderers ha realizado un pase de avance largo más que el Manchester City.